In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.DataFrame({
    'Outlook': ['Sunny','Sunny','Overcast','Rain','Rain','Rain','Overcast','Sunny'],
    'Temperature': ['Hot','Hot','Hot','Mild','Cool','Cool','Mild','Cool'],
    'Humidity': ['High','High','High','High','Normal','Normal','Normal','High'],
    'Play': ['No','No','Yes','Yes','Yes','No','Yes','No']
})

print(data)

    Outlook Temperature Humidity Play
0     Sunny         Hot     High   No
1     Sunny         Hot     High   No
2  Overcast         Hot     High  Yes
3      Rain        Mild     High  Yes
4      Rain        Cool   Normal  Yes
5      Rain        Cool   Normal   No
6  Overcast        Mild   Normal  Yes
7     Sunny        Cool     High   No


In [3]:
classes,count = np.unique(data["Outlook"],return_counts=True)

In [4]:
def gini(y):
    classes, counts = np.unique(y, return_counts=True)
    probabilities = counts / counts.sum()
    gini_value = 1 - np.sum(probabilities ** 2)
    return gini_value

In [5]:
gini(data['Play'])

np.float64(0.5)

In [6]:
def gini_split(data, feature, target):

    total = len(data)
    values = data[feature].unique()

    weighted_gini = 0

    for v in values:

        subset = data[data[feature] == v]

        weight = len(subset) / total

        weighted_gini += weight * gini(subset[target])

    return weighted_gini

In [7]:
def best_feature(data, features, target):

    gini_values = {}
    for f in features:
        gini_values[f] = gini_split(data, f, target)

    return min(gini_values, key=gini_values.get)

In [8]:
best_feature(data, ['Outlook','Temperature','Humidity'],'Play')

'Outlook'

In [9]:
def build_tree(data, features, target):

    # If node is pure
    if len(data[target].unique()) == 1:
        return data[target].iloc[0]

    # If no features left
    if len(features) == 0:
        return data[target].mode()[0]

    best = best_feature(data, features, target)

    tree = {best:{}}

    for value in data[best].unique():

        subset = data[data[best] == value]

        remaining = [f for f in features if f != best]

        tree[best][value] = build_tree(subset, remaining, target)

    return tree

In [10]:
features = ['Outlook','Temperature','Humidity']

tree = build_tree(data, features, 'Play')

print(tree)

{'Outlook': {'Sunny': 'No', 'Overcast': 'Yes', 'Rain': {'Temperature': {'Mild': 'Yes', 'Cool': {'Humidity': {'Normal': 'No'}}}}}}


In [11]:
import pandas as pd
import numpy as np

# Example dataset
data = pd.DataFrame({
    'Outlook':['Sunny','Sunny','Overcast','Rain','Rain','Rain','Overcast','Sunny'],
    'Temperature':['Hot','Hot','Hot','Mild','Cool','Cool','Mild','Cool'],
    'Humidity':['High','High','High','High','Normal','Normal','Normal','High'],
    'Play':['No','No','Yes','Yes','Yes','No','Yes','No']
})


# -----------------------------
# 1. Gini of a node
# -----------------------------
def gini(y):
    classes, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return 1 - np.sum(p**2)


# -----------------------------
# 2. Gini after splitting feature
# -----------------------------
def gini_split(data, feature, target):

    total = len(data)
    values = data[feature].unique()

    gini_feature = 0

    for v in values:

        subset = data[data[feature] == v]

        weight = len(subset) / total

        gini_feature += weight * gini(subset[target])

    return gini_feature


# -----------------------------
# 3. Select best attribute
# -----------------------------
def best_attribute(data, features, target):

    gini_values = {}

    for f in features:
        gini_values[f] = gini_split(data, f, target)

    return min(gini_values, key=gini_values.get)


# -----------------------------
# 4. Build ID3 Tree
# -----------------------------
def build_tree(data, features, target):

    # if node is pure
    if len(data[target].unique()) == 1:
        return data[target].iloc[0]

    # if no features left
    if len(features) == 0:
        return data[target].mode()[0]

    best = best_attribute(data, features, target)

    tree = {best:{}}

    for value in data[best].unique():

        subset = data[data[best] == value]

        remaining = [f for f in features if f != best]

        tree[best][value] = build_tree(subset, remaining, target)

    return tree


# -----------------------------
# 5. Train the tree
# -----------------------------
features = ['Outlook','Temperature','Humidity']

tree = build_tree(data, features, 'Play')

print(tree)

{'Outlook': {'Sunny': 'No', 'Overcast': 'Yes', 'Rain': {'Temperature': {'Mild': 'Yes', 'Cool': {'Humidity': {'Normal': 'No'}}}}}}


In [12]:
from graphviz import Digraph

def draw_tree(tree):

    dot = Digraph()

    def add_nodes(tree, parent=None, edge_label=""):
        
        if isinstance(tree, dict):
            for feature, branches in tree.items():
                
                node_id = str(id(feature))
                dot.node(node_id, feature)
                
                if parent:
                    dot.edge(parent, node_id, label=edge_label)
                
                for value, subtree in branches.items():
                    add_nodes(subtree, node_id, str(value))
        
        else:
            leaf_id = str(id(tree))
            dot.node(leaf_id, str(tree), shape="box")
            dot.edge(parent, leaf_id, label=edge_label)

    add_nodes(tree)

    return dot

In [13]:
draw_tree(tree)

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

In [14]:
tree = {
    "Outlook": {
        "Sunny": "No",
        "Overcast": "Yes",
        "Rain": {
            "Humidity": {
                "High": "Yes",
                "Normal": "No"
            }
        }
    }
}

graph = draw_tree(tree)
graph.render("decision_tree", view=True)

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

In [ ]:
pip install graphviz

In [ ]:
import graphviz
graphviz.Digraph()

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH